In [41]:
import boto3
import json
import pandas as pd
import time
from tqdm import tqdm
import re
import os

# Iniciar clientes S3 e SageMaker
s3_client = boto3.client('s3', region_name='eu-west-1')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name="eu-west-1")

bucket_name = 'i32419'

# Leitura do JSON já no S3
obj = s3_client.get_object(Bucket=bucket_name, Key='datasets/synthetic_booking_emails.json')
json_content = obj['Body'].read().decode('utf-8')
data = json.loads(json_content)
df = pd.DataFrame(data)

endpoint_name = 'meta-textgenerationneuron-llama-3-2-1b-2025-07-11-20-51-32-569'

example_indices = [0, 311, 12, 206, 418]


def montar_prompt_llama32_few_shot(df, target_index, example_indices):
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are an assistant that extracts car rental booking details from emails.\n"
        "Your job is to read each email and return EXACTLY 4 lines in the following format:\n"
        "- Customer name: [name]\n"
        "- Car model: [model]\n"
        "- Pickup: [datetime], [location]\n"
        "- Dropoff: [datetime], [location]\n\n"
        "Do not include any extra text\n"
        "<|eot_id|>\n"
    )

    for idx in example_indices:
        email = df.loc[idx, "body"]
        # Preenche manualmente as respostas esperadas para os exemplos
        if idx == 0:
            resposta = (
                "Customer name: Inês Santos\n"
                "Car model: Ford Fiesta\n"
                "Pickup: 2025-11-18 11:15 em Gaia Station\n"
                "Dropoff: 2025-11-22 19:00 em Santa Cruz Downtown"
            )
        elif idx == 311:
            resposta = (
                "Customer name: John Garcia\n"
                "Car model: Peugeot 208\n"
                "Pickup: 2026-05-23 09:45 (Faro Airport)\n"
                "Dropoff: 2026-05-28 09:30 (Gaia Station)"
            )
        elif idx == 206:
            resposta = (
                "Customer name: Miguel Martins\n"
                "Car model: Renault Clio\n"
                "Pickup: 2026-01-26 12:45 em Lisbon Airport\n"
                "Dropoff: 2026-01-29 10:45 em Porto Airport"
            )

        elif idx == 108:
            resposta = (
                "Customer name: John Silva\n"
                "Car model: Renault Clio\n"
                "Pickup: 2026-01-22 18:00 at Funchal Airport\n"
                "Dropoff: 2026-01-29 15:45 at Porto Airport"
            )

        elif idx == 418:
            resposta = (
                "Customer name: Carlos Martins\n"
                "Car model: Toyota Yaris\n"
                "Pickup: 2025-08-16 12:30 at Lisbon Airport\n"
                "Dropoff: 2025-08-25 13:30 at Santa Cruz Downtown"
            )

        prompt += (
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{email.strip()}\n"
            "<|eot_id|>\n"
            "<|start_header_id|>assistant<|end_header_id|>\n"
            f"{resposta}\n"
            "<|eot_id|>\n"
        )

    # Exemplo alvo
    email = df.loc[target_index, "body"]
    prompt += (
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{email.strip()}\n"
        "<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

    return prompt


def invoke_prompt_endpoint(prompt, max_tokens=50):
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps({
            'inputs': prompt,
            'parameters': {
                'max_new_tokens': max_tokens,
                'temperature': 0.1,
                'top_p': 0.1
            }
        })
    )
    result = response['Body'].read().decode('utf-8').strip()
    # O resultado vem em JSON, com o texto gerado na chave 'generated_text'
    result_json = json.loads(result)
    generated = result_json.get('generated_text', '').strip()
    return generated


def invoke_prompt_endpoint_with_retry(prompt, max_tokens=50, retries=3, delay=5):
    for attempt in range(retries):
        try:
            return invoke_prompt_endpoint(prompt, max_tokens)
        except Exception as e:
            print(f"Tentativa {attempt+1} falhou: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise


def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")


def salvar_checkpoint(results, local_file="synthentic_booking_email_few_shots.json"):
    results_df = pd.DataFrame(results)
    results_df.to_json(local_file, orient="records", indent=4, force_ascii=False)
    upload_file(local_file, f'output/{local_file}')


results = []
local_checkpoint_file = "synthentic_booking_email_few_shots.json"

# Retoma o checkpoint se existir
if os.path.exists(local_checkpoint_file):
    with open(local_checkpoint_file, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Retomando de checkpoint com {len(results)} registros.")
    processed_ids = {r['email_id'] for r in results}
else:
    processed_ids = set()


for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in example_indices:
        continue  # exemplos few-shot
    email_id = row.get('email_id', idx)
    if email_id in processed_ids:
        continue  # já processado

    prompt_text = montar_prompt_llama32_few_shot(df, idx, example_indices)
    try:
        response_text = invoke_prompt_endpoint_with_retry(prompt_text)
    except Exception as e:
        print(f"Erro na linha {idx}: {e}")
        response_text = None

    print(f"Email ID: {email_id}")
    print(f"Resposta gerada:\n{response_text}")
    print("-" * 50)

    results.append({
        'email_id': email_id,
        'raw_response': response_text or ""
    })

    # Checkpoint a cada 100 exemplos
    if len(results) % 100 == 0:
        print(f"Checkpoint salvo com {len(results)} registros.")
        salvar_checkpoint(results)

    time.sleep(1)  # pequeno delay entre chamadas


# Salvar ao final também
salvar_checkpoint(results)  # pequeno delay entre chamadas

  0%|          | 0/500 [00:00<?, ?it/s]

Email ID: rentalcars_198246
Resposta gerada:
Customer name: John Silva
Car model: Hyundai i20
Pickup: 2025-10-20 16:00 em Gaia Station
Dropoff: 2025-10-24 16:15 em Santa Cruz Downtown
--------------------------------------------------


  0%|          | 2/500 [00:02<10:50,  1.31s/it]

Email ID: rentalcars_948749
Resposta gerada:
Customer name: Tiago Smith
Car model: Toyota Yaris
Pickup: 2025-07-04 20:15 em Porto Airport
Dropoff: 2025-07-17 19:45 em Lisbon Airport
--------------------------------------------------


  1%|          | 3/500 [00:05<15:01,  1.81s/it]

...

Email ID: direct_booking_742917
Resposta gerada:
Customer name: Sara Smith
Car model: Ford Fiesta
Pickup: 2025-07-28 16:15 at Funchal Airport
Dropoff: 2025-08-09 15:45 at Gaia Station
--------------------------------------------------


100%|█████████▉| 499/500 [20:47<00:02,  2.52s/it]

Email ID: direct_booking_672949
Resposta gerada:
Customer name: Tiago Martins
Car model: Toyota Yaris
Pickup: 2025-09-08 14:30 at Lisbon Airport
Dropoff: 2025-09-15 15:00 at Faro Airport
--------------------------------------------------


100%|██████████| 500/500 [20:49<00:00,  2.50s/it]

Arquivo synthentic_booking_email_few_shots.json enviado para s3://i32419/output/synthentic_booking_email_few_shots.json


In [7]:
# Avaliação

# Carrega os arquivos locais
df_body = pd.read_json("synthetic_booking_emails.json")
df_llm = pd.read_json("synthentic_booking_email_few_shots.json")


def extract_fields_from_body(text):
    # Nome
    match_nome = re.search(
        r"(?:Caro\(a\)|c|Olá|Dear|Hello)\s+([^\n,]+)",
        text,
        re.IGNORECASE
    )

    # Modelo do carro
    match_modelo = re.search(
        r"^(?:Viatura|Vehicle|Car)\s*[:\-–]\s*(.+)$",
        text,
        re.IGNORECASE | re.MULTILINE
    )

    # Pick-up (captura data e localização com ou sem parênteses)
    match_pickup = re.search(
        r"(?:levantamento|Levantar|Pick(?:[-–]?)up(?: date)?|Data de levantamento)[: ]+"
        r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}(?: \([^\)]+\)|(?: em | at )[^\n]+)?)",
        text,
        re.IGNORECASE
    )

    # Drop-off (captura data e localização com ou sem parênteses)
    match_dropoff = re.search(
        r"(?:devolução|Devolver|Drop(?:[-–]?)off(?: date)?|Return|Data de devolução)[: ]+"
        r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}(?: \([^\)]+\)|(?: em | at )[^\n]+)?)",
        text,
        re.IGNORECASE
    )

    return {
        "Customer name": match_nome.group(1).strip() if match_nome else "",
        "Car model": match_modelo.group(1).strip() if match_modelo else "",
        "Pickup": match_pickup.group(1).strip() if match_pickup else "",
        "Dropoff": match_dropoff.group(1).strip() if match_dropoff else ""
    }


def parse_llm_flat_response(text):
    aliases = {
        "Customer name": ["Customer name:"],
        "Car model": ["Car model:"],
        "Pickup": ["Pickup:", "Levantar:"],
        "Dropoff": ["Dropoff:", "Devolver:"]
    }

    result = {key: "" for key in aliases}

    for key, variantes in aliases.items():
        for variante in variantes:
            idx = text.find(variante)
            if idx != -1:
                start = idx + len(variante)
                # procurar início do próximo campo
                remaining_aliases = sum(aliases.values(), [])
                next_positions = [text.find(v, start) for v in remaining_aliases if text.find(v, start) != -1]
                end = min(next_positions) if next_positions else len(text)
                result[key] = text[start:end].strip()
                break  # não precisa procurar outras variantes

    return result


# Aplicar regex ao body
df_body_extracted = df_body.copy()
df_body_extracted = df_body_extracted[["email_id", "body"]]
df_body_extracted = df_body_extracted[df_body_extracted["email_id"].isin(df_llm["email_id"])]

extracted = df_body_extracted["body"].apply(extract_fields_from_body)
extracted_df = pd.json_normalize(extracted)
df_regex = pd.concat([df_body_extracted["email_id"].reset_index(drop=True), extracted_df], axis=1)

# Aplicar a função parse_llm_flat_response pois a resposta do modelo veio apenas em 1 linha (raw_response)
df_llm["parsed"] = df_llm["raw_response"].apply(parse_llm_flat_response)
df_llm = pd.concat([df_llm.drop(columns=["raw_response"]), pd.json_normalize(df_llm["parsed"])], axis=1)

# Juntar com as respostas do LLM para comparação
df_compare = df_llm.merge(df_regex, on="email_id", suffixes=("_llm", "_regex"))

# Métricas
field_names = ["Customer name", "Car model", "Pickup", "Dropoff"]
results = {}
total = len(df_compare)


# Função de comparação que trata "at" e "em" como equivalentes para Pickup e Dropoff (porque o modelo coloca alguns at no lugar de em's apesar da extração estar bem feita)
def compare_at_em_equal(val1, val2, field):
    if field in ["Pickup", "Dropoff"]:
        if not isinstance(val1, str) or not isinstance(val2, str):
            return val1 == val2
        # Substitui temporariamente " at " por " em " para comparação
        val1_norm = re.sub(r"\s+at\s+", " em ", val1, flags=re.IGNORECASE)
        val2_norm = re.sub(r"\s+at\s+", " em ", val2, flags=re.IGNORECASE)
        return val1_norm == val2_norm
    return val1 == val2


for field in field_names:
    acc = df_compare.apply(
        lambda row: compare_at_em_equal(row[f"{field}_llm"], row[f"{field}_regex"], field),
        axis=1
    ).mean()
    results[field] = round(acc * 100, 2)

df_compare["exact_match"] = df_compare.apply(
    lambda row: all(compare_at_em_equal(row[f"{field}_llm"], row[f"{field}_regex"], field) for field in field_names),
    axis=1
)

results["Exact Match"] = round(df_compare["exact_match"].mean() * 100, 2)

# Criar colunas indicando se cada campo bateu (True/False)
for field in field_names:
    df_compare[f"{field}_match"] = df_compare.apply(
        lambda row: compare_at_em_equal(row[f"{field}_llm"], row[f"{field}_regex"], field),
        axis=1
    )


# Visualizar diferenças:

# Filtrar só as linhas onde pelo menos um campo divergiu
diff_rows = df_compare[~df_compare[[f"{field}_match" for field in field_names]].all(axis=1)]

# Guardar as diferenças para análise
diff_report_columns = ["email_id"]
for field in field_names:
    diff_report_columns += [f"{field}_llm", f"{field}_regex", f"{field}_match"]

# Salvar relatório detalhado das diferenças
diff_rows[diff_report_columns].to_csv("diferencas_detectadas.csv", index=False)

print(f"Diferenças encontradas: {len(diff_rows)} linhas com pelo menos um campo divergente.")

# Guardar relatório
report_text = "\n".join([f"{k}: {v}%" for k, v in results.items()])
with open("relatorio_metricas_few_shots_emails.txt", "w", encoding="utf-8") as f:
    f.write("Relatório de Avaliação - Comparação entre Regex e Modelo\n")
    f.write("="*50 + "\n\n")
    f.write(report_text)


# Enviar relatório para o S3
upload_file("relatorio_metricas_few_shots_emails.txt", "output/relatorio_metricas_few_shots_emails.txt")
upload_file("diferencas_detectadas.csv", "output/diferencas_detectadas.csv")

Diferenças encontradas: 276 linhas com pelo menos um campo divergente.
Arquivo relatorio_metricas_few_shots_emails.txt enviado para s3://i32419/output/relatorio_metricas_few_shots_emails.txt
Arquivo diferencas_detectadas.csv enviado para s3://i32419/output/diferencas_detectadas.csv
